# Semantic roundtrip: study analysis

One notebook with two explicit modes. Restart the kernel and use **Run All**.

- `ANALYSIS_MODE=full_study` (default): set the three core job paths and completed supplements.
- `ANALYSIS_MODE=style_pilot`: review only; set `FREE_JOB`, `SKETCH_JOB`, `COMIC_JOB`,
  `PHOTOREALISTIC_JOB`, optionally `RATING_JOB`, and `OUTPUT_DIR`. Sections A–E do not execute.

The full-study sections remain unchanged:

- **A — Direct:** complete 4×4 PG/BI matrix and four paired-route baselines.
- **B — Indirect:** complete 3×4×3 PG/BB/BI matrix from local and Aqueduct jobs.
- **C — RQ4:** domain comparison.
- **D — Supplements:** sketch, comic, native thinking and high illustratability.
- **E — Technical validity:** coverage, failures and supporting accuracy for all loaded matrices.

The Aqueduct job must inherit from the selected local indirect job. The thinking job must
inherit from the selected direct job. Do not combine unrelated jobs.

Terminal failures are observations, not exclusions. The primary metric keeps every planned
observation in its denominator. Supplement paths may be omitted when that analysis is not yet
available.

## Preparation

Set the paths below or the corresponding environment variables before launching Jupyter.
Use absolute outer job-directory paths containing `job_state.sqlite`. Use a separate output
directory for each report.

In full-study mode, four seed observations are averaged **per title** before the primary analysis.
The optional style pilot uses one prompt seed × one image seed (one observation per title/pair). End-to-end
Strict Exact Match is primary: verifier rejections, missing predictions and terminal failures
score zero. Prediction-only Strict Exact Match is reported separately as correct predictions
divided by available predictions. Contrasts use 10,000 paired whole-title bootstrap resamples,
stratified only by domain (seed 20260829). Intervals are pointwise 95% percentile intervals.
Accuracy is shown in percent; differences are percentage points (pp).

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib.ticker import MaxNLocator

from semantic_roundtrip.analysis import (
    aggregate_titles,
    illustratability_spearman,
    load_job,
    paired_stratified_bootstrap,
)
from semantic_roundtrip.evaluation import EXACT_MATCH_METHOD, NORMALIZED_EXACT_METHOD

# Example: export DIRECT_JOB="/absolute/path/to/20260902T120000Z_final-direct-core_ab12cd34"
# Every variable below uses the same outer job-directory form, not a child run or SQLite file.
ANALYSIS_MODE = os.getenv("ANALYSIS_MODE", "full_study")
if ANALYSIS_MODE not in {"full_study", "style_pilot"}:
    raise ValueError("ANALYSIS_MODE must be full_study or style_pilot.")
FREE_JOB = os.getenv("FREE_JOB") or None
PHOTOREALISTIC_JOB = os.getenv("PHOTOREALISTIC_JOB") or None
RATING_JOB = os.getenv("RATING_JOB") or None
DIRECT_JOB = os.getenv("DIRECT_JOB") or None
INDIRECT_JOB = os.getenv("INDIRECT_JOB") or None
AQUEDUCT_JOB = os.getenv("AQUEDUCT_JOB") or None
SKETCH_JOB = os.getenv("SKETCH_JOB") or None
COMIC_JOB = os.getenv("COMIC_JOB") or None
THINKING_JOB = os.getenv("THINKING_JOB") or None
ILLUSTRATABLE_JOB = os.getenv("ILLUSTRATABLE_JOB") or None
OUTPUT_DIR = (
    Path(
        os.getenv(
            "OUTPUT_DIR",
            "results/style_pilot" if ANALYSIS_MODE == "style_pilot" else "results",
        )
    )
    .expanduser()
    .resolve()
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message="The behavior of DataFrame concatenation with empty or all-NA entries is deprecated.*",
)
sns.set_theme(
    style="whitegrid",
    context="notebook",
    rc={"pdf.fonttype": 42, "ps.fonttype": 42, "axes.unicode_minus": False},
)

QG = ["q25", "g3", "q38", "g4"]
LOCAL = ["d32", "o120"]
TEXT = [*LOCAL, "v4"]
PAIRS = {
    "Qwen": ("q25", "q38"),
    "Gemma": ("g3", "g4"),
    "Family 2025": ("q25", "g3"),
    "Family 2026": ("q38", "g4"),
}
DOMAINS = {
    "songs": ("#0072B2", "o"),
    "movies": ("#D55E00", "s"),
    "bands": ("#009E73", "^"),
}
METRIC = "end_to_end_strict_accuracy"
TITLE_KEYS = ["dataset_id", "item_key", "domain", "title_length_group"]
RATING_KEYS = ["pg", *TITLE_KEYS]
INTERVAL_COLUMNS = ["estimate", "ci95_low", "ci95_high"]
pd.options.display.float_format = "{:.2f}".format

### Shared preparation and plotting helpers

These helpers do not select or load jobs. Accuracy heatmaps use the primary end-to-end metric
on a 0–100% scale; difference heatmaps use a shared symmetric -100 to +100 pp scale. Grey means
a matrix cell is absent, not zero. Effect plots show estimates and the already calculated,
pointwise paired 95% intervals. Figures appear below their cells and are also saved as
300-dpi PNGs and vector PDFs in `OUTPUT_DIR`, with fixed scientific titles. V4 is an externally hosted, larger configuration,
not an equal-size or equal-compute comparison.

In [ ]:
def annotate(frame):
    roles = frame["entry_name"].str.extract(
        r"^(?:direct|indirect)_pg_(?P<pg>[^_]+)(?:_bb_(?P<bb>[^_]+))?_bi_(?P<bi>[^_]+)$"
    )
    return frame.join(roles)


def difference(positive, negative):
    positive, negative = list(positive), list(negative)
    return {c: 1 / len(positive) for c in positive} | {
        c: -1 / len(negative) for c in negative
    }


def effects(frame, contrasts, condition_column="condition"):
    rows = []
    for domain, subset in [("all", frame), *frame.groupby("domain", sort=True)]:
        for comparison, weights in contrasts.items():
            result = paired_stratified_bootstrap(
                subset,
                condition_weights=weights,
                condition_column=condition_column,
            )
            rows.append({"domain": domain, "comparison": comparison, **result})
    table = pd.DataFrame(rows).rename(columns={"effect": "estimate"})
    table[INTERVAL_COLUMNS] *= 100
    return table


def save(fig, name, title):
    fig.suptitle(title, fontsize=14, fontweight="bold")
    fig.savefig(OUTPUT_DIR / f"{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / f"{name}.pdf", bbox_inches="tight")
    plt.show()
    plt.close(fig)

In [ ]:
def heatmap(ax, frame, models, title, baseline=None):
    values = (100 * frame.groupby(["pg", "bi"])[METRIC].mean().unstack()).reindex(
        index=models,
        columns=models,
    )
    labels = [m.upper() + ("*" if m == "v4" else "") for m in models]
    if baseline is None:
        cmap = sns.color_palette("blend:#f7fbff,#6baed6", as_cmap=True)
    else:
        reference = (
            100 * baseline.groupby(["pg", "bi"])[METRIC].mean().unstack()
        ).reindex(index=models, columns=models)
        values = values - reference
        cmap = "vlag"
    ax.set_facecolor("#dddddd")
    sns.heatmap(
        values,
        ax=ax,
        annot=True,
        fmt=".1f" if baseline is None else "+.1f",
        vmin=0 if baseline is None else -100,
        vmax=100,
        cmap=cmap,
        cbar=False,
        square=True,
        linewidths=0.5,
        linecolor="white",
        xticklabels=labels,
        yticklabels=labels,
    )
    ax.set(xlabel="BI", ylabel="PG", title=title)
    return ax.collections[0]


def bb_heatmaps(frame, models, name, title):
    fig, axes = plt.subplots(2, 2, figsize=(9, 7), layout="constrained")
    for bb, ax in zip(QG, axes.flat):
        image = heatmap(ax, frame[frame.bb == bb], models, f"BB = {bb.upper()}")
    fig.colorbar(image, ax=list(axes.flat), label="End-to-end Strict Exact Match (%)")
    save(fig, name, title)


def interval_plot(
    ax, table, xlabel="Difference (pp), paired 95% CI", zero=True, colors=None
):
    for position, row in enumerate(table.itertuples()):
        color = colors[position] if colors is not None else "#0072B2"
        if pd.isna(row.estimate):
            ax.text(0.5, position, "n/a", transform=ax.get_yaxis_transform())
            continue
        has_interval = pd.notna(row.ci95_low) and pd.notna(row.ci95_high)
        if has_interval:
            ax.hlines(position, row.ci95_low, row.ci95_high, color=color, linewidth=1.8)
            ax.vlines(
                [row.ci95_low, row.ci95_high],
                position - 0.045,
                position + 0.045,
                color=color,
                linewidth=1.5,
            )
        ax.plot(row.estimate, position, "o", color=color, markersize=6)
        value = f"{row.estimate:+.1f}" if zero else f"{row.estimate:.1f}"
        ax.annotate(
            value if has_interval else f"{value} (CI n/a)",
            (row.estimate, position),
            xytext=(0, 9),
            textcoords="offset points",
            ha="center",
            color=color,
            fontsize=9,
            fontweight="bold",
        )
    ax.set(
        yticks=range(len(table)),
        yticklabels=table.comparison.str.replace("−", "-", regex=False),
        xlabel=xlabel,
        ylim=(len(table) - 0.5, -0.6),
    )
    if zero:
        ax.axvline(0, color="#555555", linestyle="--", linewidth=1)
    ax.margins(x=0.12)
    ax.grid(axis="x", color="#D9D9D9", linewidth=0.8)
    ax.grid(axis="y", visible=False)
    sns.despine(ax=ax)


def domain_overview(frame, name):
    conditions = frame.condition.unique()
    table = effects(
        frame, {"Mean accuracy": dict.fromkeys(conditions, 1 / len(conditions))}
    )
    table = table[table.domain != "all"].assign(comparison=lambda f: f.domain)
    fig, ax = plt.subplots(figsize=(6, 3), layout="constrained")
    interval_plot(ax, table, "End-to-end Strict Exact Match (%), 95% CI", zero=False)
    ax.set_xlim(0, 100)
    save(fig, name, "Reconstruction accuracy by domain")
    return table

### Indirect comparisons

The six predeclared local effects are estimated on the 2×4×2 D32/O120 submatrix.
The two hosted extensions compare V4 with mean(D32,O120) over the complete 3×4×3 matrix.
This preserves the original local estimands while presenting one complete matrix. Matching
contrasts compare diagonal with off-diagonal cells and are reported separately by scope.


In [ ]:
def indirect_contrasts(frame):
    cells = (
        frame[["condition", "pg", "bb", "bi"]].drop_duplicates().set_index("condition")
    )
    contrasts = {}
    for role in ["pg", "bb", "bi"]:
        pairs = (
            [(f"{y.upper()} − {x.upper()}", [y], [x]) for x, y in PAIRS.values()]
            if role == "bb"
            else [
                ("O120 − D32", ["o120"], ["d32"]),
            ]
        )
        if role != "bb" and "v4" in cells[role].values:
            pairs.append(("V4 − mean(D32, O120)", ["v4"], LOCAL))
        for label, positive, negative in pairs:
            contrasts[f"{role.upper()}: {label}"] = difference(
                cells.index[cells[role].isin(positive)],
                cells.index[cells[role].isin(negative)],
            )
    return contrasts


def matching_contrasts(frame):
    cells = (
        frame[["condition", "pg", "bb", "bi"]].drop_duplicates().set_index("condition")
    )
    contrasts = {}
    for label, subset in [
        ("All BB", cells),
        *[(f"BB={bb.upper()}", cells[cells.bb == bb]) for bb in QG],
    ]:
        contrasts[label] = difference(
            subset.index[subset.pg == subset.bi], subset.index[subset.pg != subset.bi]
        )
    return contrasts

### Exploratory illustratability

One point is one title. Ratings are compared with mean reconstruction accuracy across all
downstream models in the selected scope. Domain colours expose possible pooled-domain
patterns; correlation is not evidence of causation or human-measured difficulty.
Missing ratings are not replaced. Constant variables give undefined correlation/intervals.


In [ ]:
def correlation_label(row):
    rho = "n/a" if pd.isna(row.spearman_rho) else f"{row.spearman_rho:.2f}"
    interval = (
        "n/a"
        if pd.isna(row.ci95_low) or pd.isna(row.ci95_high)
        else f"[{row.ci95_low:.2f}, {row.ci95_high:.2f}]"
    )
    return f"ρ={rho}, 95% CI {interval}"


def rating_analysis(ratings, frame, name, title):
    ratings = ratings.reindex(columns=["entry_name", *TITLE_KEYS, "score"]).copy()
    ratings["pg"] = ratings.entry_name.astype("string").str.extract(
        r"_pg_([^_]+)_", expand=False
    )
    outcomes = frame.groupby(RATING_KEYS, as_index=False)[METRIC].mean()
    models = [m for m in QG + TEXT if m in outcomes.pg.values]
    table = illustratability_spearman(ratings, outcomes).set_index("pg").reindex(models)
    table["titles"] = table.titles.astype("Int64").fillna(0)
    table["missing_ratings"] = outcomes.groupby("pg").size() - table.titles
    pairs = outcomes.merge(ratings[[*RATING_KEYS, "score"]], on=RATING_KEYS).dropna(
        subset=["score"]
    )
    pairs["accuracy_percent"] = 100 * pairs[METRIC]
    fig, axes = plt.subplots(
        1,
        len(models),
        figsize=(3.5 * len(models), 4),
        squeeze=False,
        layout="constrained",
    )
    for model, ax in zip(models, axes.flat):
        sns.scatterplot(
            data=pairs[pairs.pg == model],
            x="score",
            y="accuracy_percent",
            hue="domain",
            style="domain",
            hue_order=list(DOMAINS),
            style_order=list(DOMAINS),
            palette={domain: style[0] for domain, style in DOMAINS.items()},
            markers={domain: style[1] for domain, style in DOMAINS.items()},
            alpha=0.7,
            ax=ax,
            legend=model == models[0],
        )
        row = table.loc[model]
        ax.set(
            title=f"{model.upper()}\n{correlation_label(row)}\nn={int(row.titles)}",
            xlabel="Illustratability (0–100)",
            ylabel="Mean title accuracy (%)",
            xlim=(-2, 102),
            ylim=(-2, 102),
        )
    axes[0, 0].legend(fontsize=8, title="Domain")
    save(fig, name, title)
    return table.reset_index(), pairs

### Diagnostic tables

Technical validity covers all loaded matrices. `end_to_end_strict_accuracy` divides correct,
verifier-accepted predictions by all planned observations. `prediction_only_strict_accuracy`
divides correct predictions by available predictions, regardless of verification.
`verifier_accepted_strict_accuracy` divides correct, accepted predictions by all accepted
images; missing predictions within that subset still score zero. Empty denominators show
`n/a`. Conditional accuracy is not an improvement on the same population.

Counts and pooled ratios are reported by condition and domain, not averaged from conditional
title means. Coverage, acceptance, rejections, missing verifier decisions, missing predictions
and missing predictions linked to an error stay separate. Verifier counts use one image
opportunity per condition rather than one per route. Stage errors cover all pipeline stages.
Imported errors and timings are deduplicated against their original local records.

Normalized Exact Match uses NFC, casefold and whitespace normalization; middle dot U+00B7
and dashes U+2010–U+2015 map to ASCII hyphen-minus. One extra matching straight or
English/German curly quote pair may wrap the prediction; other punctuation stays intact.
Its tables retain equal title weighting. Full-study primary figures and intervals use
Strict Exact Match; the review-only style pilot shows both metrics separately.
Both metric method identifiers are printed with the supporting tables.

Answer likelihood is based only on visible answer-token log probabilities. It is diagnostic,
not calibrated confidence and not suitable for ranking model families.

In [ ]:
def accuracy_rates(
    planned, predictions, end_to_end_correct, prediction_correct, accepted
):
    return pd.DataFrame(
        {
            "end_to_end_strict_accuracy": 100 * end_to_end_correct / planned,
            "prediction_only_strict_accuracy": (
                100 * prediction_correct / predictions.replace(0, np.nan)
            ),
            "verifier_accepted_strict_accuracy": (
                100 * end_to_end_correct / accepted.replace(0, np.nan)
            ),
            "coverage": 100 * predictions / planned,
            "acceptance_rate": 100 * accepted / planned,
        }
    )


_control = accuracy_rates(
    pd.Series([4]), pd.Series([3]), pd.Series([1]), pd.Series([2]), pd.Series([2])
).iloc[0]
assert np.allclose(_control.to_numpy(), [25.0, 100 * 2 / 3, 50.0, 75.0, 50.0])


def technical_tables(observation_sets, title_sets, job_sets):
    tables = {}
    accuracy_rows, verifier_rows = [], []
    for study, observations in observation_sets.items():
        scoped = pd.concat([observations, observations.assign(domain="all")])
        scoped = pd.concat([scoped, scoped.assign(condition="all")])
        grouped = (
            scoped.groupby(["condition", "route", "domain"], sort=True)
            .agg(
                planned_observations=("image_seed", "size"),
                predictions=("prediction_id", "count"),
                prediction_correct=(
                    "strict_exact_match",
                    lambda s: int(s.astype("boolean").fillna(False).sum()),
                ),
                end_to_end_correct=("end_to_end_strict_score", "sum"),
                accepted=(
                    "verification_passed",
                    lambda s: int(s.eq(True).sum()),
                ),
                verifier_rejections=(
                    "verification_passed",
                    lambda s: int(s.eq(False).sum()),
                ),
                missing_verifier_decisions=(
                    "verification_passed",
                    lambda s: int(s.isna().sum()),
                ),
                missing_predictions_with_error=(
                    "prediction_status",
                    lambda s: int(s.eq("failed").sum()),
                ),
            )
            .reset_index()
        )
        grouped.insert(0, "study", study)
        grouped["missing_predictions"] = (
            grouped.planned_observations - grouped.predictions
        )
        grouped[
            [
                "end_to_end_strict_accuracy",
                "prediction_only_strict_accuracy",
                "verifier_accepted_strict_accuracy",
                "coverage",
                "acceptance_rate",
            ]
        ] = accuracy_rates(
            grouped.planned_observations,
            grouped.predictions,
            grouped.end_to_end_correct,
            grouped.prediction_correct,
            grouped.accepted,
        )
        accuracy_rows.append(grouped)

        images = observations.drop_duplicates(
            ["condition", "dataset_id", "item_key", "prompt_seed", "image_seed"]
        ).assign(
            passed=lambda f: f.verification_passed.eq(True),
            rejected=lambda f: f.verification_passed.eq(False),
        )
        verifier_rows.append(
            {
                "study": study,
                "condition_images": len(images),
                "materialized": int(images.image_id.notna().sum()),
                "verified": int(images.verification_passed.notna().sum()),
                "passed": int(images.passed.sum()),
                "rejected": int(images.rejected.sum()),
                "missing_decisions": int(images.verification_passed.isna().sum()),
            }
        )
    tables["accuracy_and_coverage_percent"] = pd.concat(
        accuracy_rows, ignore_index=True
    )
    tables["verifier"] = pd.DataFrame(verifier_rows)

    supporting_metrics = [
        "end_to_end_normalized_accuracy",
        "prediction_only_normalized_accuracy",
    ]
    supporting = []
    for study, scores in title_sets.items():
        table = (100 * scores.groupby("route")[supporting_metrics].mean()).reset_index()
        table.insert(0, "study", study)
        supporting.append(table)
    tables["title_weighted_normalized_metrics_percent"] = pd.concat(
        supporting, ignore_index=True
    )

    produced = pd.concat(observation_sets.values(), ignore_index=True)
    produced = produced[produced.prediction_id.notna()].assign(
        answer_likelihood=lambda f: f.confidence.astype(float),
    )
    likelihood = (
        produced.groupby(["prediction_model", "route"], dropna=False)
        .agg(
            predictions=("prediction_id", "count"),
            available=("answer_likelihood", "count"),
            median=("answer_likelihood", "median"),
            q25=("answer_likelihood", lambda s: s.quantile(0.25)),
            q75=("answer_likelihood", lambda s: s.quantile(0.75)),
        )
        .reset_index()
    )
    likelihood["availability_percent"] = (
        100 * likelihood.available / likelihood.predictions
    )
    tables["answer_likelihood"] = likelihood

    errors, timings = [], []
    for study, job in job_sets:
        errors.append(job.errors.assign(study=study))
        timings.append(job.timings.assign(study=study))
    errors = pd.concat(errors, ignore_index=True).reindex(
        columns=[
            "study",
            "provenance_error_key",
            "execution_origin",
            "stage",
            "error_type",
            "terminal_error",
            "recovered",
        ]
    )
    errors["origin_order"] = errors.execution_origin.map(
        {"local": 0, "imported": 1}
    ).fillna(2)
    errors = errors.sort_values("origin_order").drop_duplicates("provenance_error_key")
    tables["error_attempts"] = (
        errors.groupby(
            ["study", "stage", "error_type", "terminal_error", "recovered"],
            dropna=False,
        )
        .size()
        .rename("attempts")
        .reset_index()
    )
    timings = pd.concat(timings, ignore_index=True).reindex(
        columns=[
            "study",
            "provenance_timing_key",
            "execution_origin",
            "kind",
            "stage",
            "action",
            "duration_seconds",
        ]
    )
    timings["origin_order"] = timings.execution_origin.map(
        {"local": 0, "imported": 1}
    ).fillna(2)
    timings = timings.sort_values("origin_order").drop_duplicates(
        "provenance_timing_key"
    )
    tables["task_time_minutes"] = (
        timings[timings.kind.eq("task") & timings.action.eq("execute")]
        .groupby(["study", "stage"], dropna=False)
        .duration_seconds.sum(min_count=1)
        .div(60)
        .rename("minutes")
        .reset_index()
    )
    return tables

## Style pilot — optional review-only analysis

This section runs only with `ANALYSIS_MODE=style_pilot`. It does not execute model jobs.
Supply available completed style jobs via `FREE_JOB`, `SKETCH_JOB`, `COMIC_JOB` and
`PHOTOREALISTIC_JOB`. Only Q25→Q25, G3→G3, Q38→Q38 and G4→G4 direct entries are loaded:
these are diagonal comparisons, **not** four complete 4×4 matrices. The paired pilot contains
90 titles (30 per domain; `configs/datasets/style_pilot_titles_v1.yaml`), separate from the
random and high-illustratability study title sets. Select the intended sources yourself; this notebook does not compare title sets or fingerprints.

For the planned 90-title pilot, both Strict and Normalized Exact Match use all 360 planned direct observations per style,
including rejected images, missing predictions and terminal failures as zero. Each title has one prompt-seed × image-seed observation; model pairs are weighted equally. Figures show overall, model-pair,
domain and model-pair-by-domain estimates with pointwise 95% domain-stratified whole-title
bootstrap intervals (10,000 resamples, seed 20260829). Counts, coverage and rejections are separate.
Prompt title hits are report-only, never exclusions; set `PROMPT_HIT_DIAGNOSTICS=0` to disable.

`RATING_JOB`, when supplied, must be the completed `candidate_illustratability` job covering
**all 900 candidates**. The snapshot roster retains candidates missing every rating.
Each candidate contributes only its equal Q25/G3/Q38/G4 mean with four valid integer ratings
from 0 to 100. The 900-candidate pool is not the 90-title pilot or main study sample.
No selected-title CSV or reconstruction-job ratings substitute for this source. If the path
is unset or unfinished, style review continues and explicitly reports the missing candidate distribution.
Conversely, `RATING_JOB` alone produces the candidate distribution without requiring style jobs.
Unset, missing or unfinished style jobs are listed as unavailable and never plotted as if observed.
Invalid completed inputs still raise an error. Use a fresh output directory for each execution.
Histograms use fixed 10-point bins on 0–100, with complete and missing counts per domain.

Tables display inline and export CSV; figures display inline and export 300-dpi PNG/vector PDF.
Actual title/observation counts are reported, not required to equal 90 at analysis time;
the pilot configuration defines the intended 90-title design. Reduced model-free fixtures are QA only.

A versioned JSON manifest records source job paths, snapshot hashes (recorded, not compared), metric method IDs,
denominators and availability. Use a separate `OUTPUT_DIR`; no source runs are changed.


In [ ]:
if ANALYSIS_MODE == "style_pilot":
    import hashlib
    import json
    import platform
    from datetime import UTC, datetime
    from importlib.metadata import version

    from IPython.display import JSON

    from semantic_roundtrip import evaluation
    from semantic_roundtrip.config_resolution import load_effective_config
    from semantic_roundtrip.evaluation import (
        PROMPT_TITLE_MATCH_METHOD,
        title_occurs_in_text,
    )
    from semantic_roundtrip.job import JOB_SNAPSHOT_FILENAME, load_job_snapshot
    from semantic_roundtrip.persistence.job.database import (
        read_job_entries,
        read_job_record,
    )
    from semantic_roundtrip.persistence.run.config_snapshot import (
        EFFECTIVE_CONFIG_FILENAME,
    )

    PILOT_VERSION = "style_pilot_review_v1"
    PILOT_STYLES = {
        "Unrestricted": FREE_JOB,
        "Sketch": SKETCH_JOB,
        "Comic": COMIC_JOB,
        "Photorealistic": PHOTOREALISTIC_JOB,
    }
    PILOT_METRICS = {
        METRIC: ("Strict", EXACT_MATCH_METHOD, "#0072B2", "o"),
        "end_to_end_normalized_accuracy": (
            "Normalized",
            NORMALIZED_EXACT_METHOD,
            "#D55E00",
            "s",
        ),
    }
    pilot_sources = []
    pilot_exports = []
    pilot_availability = []
    prompt_hit_method = PROMPT_TITLE_MATCH_METHOD
    prompt_hit_status = "unavailable: no completed styles"

    def pilot_export(table, name):
        filename = f"style_pilot_{name}.csv"
        table.to_csv(OUTPUT_DIR / filename, index=False, na_rep="n/a")
        pilot_exports.append(filename)
        display(table)

    def pilot_save(fig, name, title):
        save(fig, f"style_pilot_{name}", f"Style pilot: {title}")
        pilot_exports.extend(f"style_pilot_{name}.{ext}" for ext in ["png", "pdf"])

    def pilot_hash(path):
        return hashlib.sha256(Path(path).read_bytes()).hexdigest()

    def pilot_load_job(path, label, entry_names):
        directory = Path(path).expanduser().resolve() if path else None
        available = {"label": label, "directory": str(directory) if directory else None}
        record = (
            read_job_record(directory / "job_state.sqlite")
            if directory and (directory / "job_state.sqlite").is_file()
            else None
        )
        available["status"] = (
            record.status
            if record
            else "path not set"
            if directory is None
            else "job not found"
        )
        pilot_availability.append(available)
        if record is None or record.status != "completed":
            print(
                f"{label}: unavailable ({available['status']}); this section is skipped."
            )
            return None, {}
        if (
            label == "Candidate pool"
            and record.name.replace("-", "_") != "candidate_illustratability"
        ):
            raise ValueError("RATING_JOB must be the candidate_illustratability job.")
        snapshot = load_job_snapshot(directory / JOB_SNAPSHOT_FILENAME)
        stored = read_job_entries(directory / "job_state.sqlite", directory)
        selected = {
            entry.name: run.run_directory
            for entry, run in zip(snapshot.entries, stored, strict=True)
            if entry.name in entry_names
        }
        job = load_job(directory, entries=entry_names)
        files = [directory / JOB_SNAPSHOT_FILENAME]
        files.extend(run / EFFECTIVE_CONFIG_FILENAME for run in selected.values())
        pilot_sources.append(
            {
                "label": label,
                "directory": str(directory),
                "job_id": record.job_id,
                "job_name": record.name,
                "status": record.status,
                "entries": {name: str(run) for name, run in selected.items()},
                "sha256": {str(path): pilot_hash(path) for path in files},
            }
        )
        return job, selected

    def pilot_scopes(frame):
        scoped = pd.concat([frame, frame.assign(domain="all")], ignore_index=True)
        return pd.concat([scoped, scoped.assign(model_pair="all")], ignore_index=True)

    diagonal_entries = [f"direct_pg_{model}_bi_{model}" for model in QG]
    pilot_jobs, pilot_observation_sets, pilot_title_sets = {}, {}, {}
    pilot_observed_counts = {}
    for style, path in PILOT_STYLES.items():
        job, _ = pilot_load_job(path, style, diagonal_entries)
        if job is None:
            continue
        observations = annotate(job.observations)
        observations = observations[observations.route.eq("direct")].copy()
        observations["style"] = style
        observations["model_pair"] = (
            observations.pg.str.upper() + " → " + observations.bi.str.upper()
        )
        scores = aggregate_titles(
            observations,
            condition_columns=["style", "pg", "bi", "model_pair"],
            expected_observations=1,
        )
        roster = observations[
            ["dataset_id", "item_key", "domain", "expected_title"]
        ].drop_duplicates()
        pilot_observed_counts[style] = {
            "titles": len(roster),
            "titles_per_domain": roster.groupby("domain").size().to_dict(),
            "planned_observations": len(observations),
        }
        pilot_jobs[style], pilot_observation_sets[style], pilot_title_sets[style] = (
            job,
            observations,
            scores,
        )
    pilot_observations = (
        pd.concat(pilot_observation_sets.values(), ignore_index=True)
        if pilot_jobs
        else pd.DataFrame()
    )
    pilot_titles = (
        pd.concat(pilot_title_sets.values(), ignore_index=True)
        if pilot_jobs
        else pd.DataFrame()
    )
    pilot_scope = {
        "available_styles": list(pilot_jobs),
        "missing_styles": [style for style in PILOT_STYLES if style not in pilot_jobs],
        "model_pairs": diagonal_entries if pilot_jobs else [],
        "planned_design_titles": 90,
        "planned_design_titles_per_domain": 30,
        "planned_design_observations_per_style": 360,
        "seed_observations_per_title": 1,
        "observed_counts": pilot_observed_counts,
    }
    print("Style pilot — review only:", pilot_scope)
    if pilot_scope["missing_styles"]:
        print(
            "No complete four-style comparison: missing",
            ", ".join(pilot_scope["missing_styles"]),
        )
    print("Strict:", EXACT_MATCH_METHOD, " | Normalized:", NORMALIZED_EXACT_METHOD)

In [ ]:
if ANALYSIS_MODE == "style_pilot" and not pilot_titles.empty:
    pilot_accuracy_rows = []
    for (style, model_pair, domain), frame in pilot_scopes(pilot_titles).groupby(
        ["style", "model_pair", "domain"], sort=False
    ):
        conditions = frame.condition.unique()
        for metric, (label, method, _, _) in PILOT_METRICS.items():
            result = paired_stratified_bootstrap(
                frame,
                condition_weights=dict.fromkeys(conditions, 1 / len(conditions)),
                metric=metric,
            )
            planned = int(frame.observations.sum())
            correct = round((frame[metric] * frame.observations).sum())
            if not np.isclose(result["effect"], correct / planned):
                raise ValueError(
                    "Title-weighted accuracy differs from the planned-denominator rate."
                )
            pilot_accuracy_rows.append(
                {
                    "style": style,
                    "model_pair": model_pair,
                    "domain": domain,
                    "metric": label,
                    "method": method,
                    "titles": result["titles"],
                    "planned_observations": planned,
                    "correct": correct,
                    "accuracy_percent": 100 * result["effect"],
                    "ci95_low": 100 * result["ci95_low"],
                    "ci95_high": 100 * result["ci95_high"],
                }
            )
    pilot_accuracy = pd.DataFrame(pilot_accuracy_rows)
    pilot_export(pilot_accuracy, "accuracy")
    pilot_export(pilot_titles, "title_scores")

    def pilot_accuracy_panel(ax, frame):
        for offset, (label, _, color, marker) in zip(
            [-0.12, 0.12], PILOT_METRICS.values()
        ):
            values = (
                frame[frame.metric.eq(label)].set_index("style").reindex(pilot_jobs)
            )
            ax.errorbar(
                np.arange(len(pilot_jobs)) + offset,
                values.accuracy_percent,
                yerr=np.vstack(
                    [
                        values.accuracy_percent - values.ci95_low,
                        values.ci95_high - values.accuracy_percent,
                    ]
                ),
                fmt=marker,
                color=color,
                capsize=2,
                markersize=5,
                label=label,
            )
        ax.set(
            xticks=range(len(pilot_jobs)),
            xticklabels=list(pilot_jobs),
            ylim=(0, 100),
            xlim=(-0.5, len(pilot_jobs) - 0.5),
        )
        ax.tick_params(axis="x", rotation=30, labelsize=9)

    pilot_domains = ["all", *DOMAINS]
    fig, axes = plt.subplots(1, 4, figsize=(16, 4.2), sharey=True, layout="constrained")
    for domain, ax in zip(pilot_domains, axes):
        pilot_accuracy_panel(
            ax,
            pilot_accuracy[
                pilot_accuracy.model_pair.eq("all") & pilot_accuracy.domain.eq(domain)
            ],
        )
        ax.set_title("Overall" if domain == "all" else domain.title())
    axes[0].set_ylabel("End-to-end accuracy (%), pointwise 95% CI")
    axes[-1].legend(loc="upper right", fontsize=8)
    pilot_save(fig, "accuracy_overall_domains", "End-to-end accuracy by domain")

    fig, axes = plt.subplots(4, 4, figsize=(17, 13), sharey=True, layout="constrained")
    for model, row in zip(QG, axes):
        pair = f"{model.upper()} → {model.upper()}"
        for domain, ax in zip(pilot_domains, row):
            pilot_accuracy_panel(
                ax,
                pilot_accuracy[
                    pilot_accuracy.model_pair.eq(pair)
                    & pilot_accuracy.domain.eq(domain)
                ],
            )
            ax.set_title(f"{pair} · {'Overall' if domain == 'all' else domain.title()}")
    axes[0, -1].legend(loc="upper right", fontsize=8)
    fig.supylabel("End-to-end accuracy (%), pointwise 95% CI")
    pilot_save(
        fig,
        "accuracy_diagonal_pairs_domains",
        "End-to-end accuracy by diagonal model pair and domain",
    )

### Style pilot — counts, coverage, rejections and optional prompt hits

Pooled `all` rows must not be added to their detail rows. Prompt counts use one prompt opportunity,
not one per image. Terminal stage-error attempts are separate from missing-prediction observations.


In [ ]:
if ANALYSIS_MODE == "style_pilot" and not pilot_titles.empty:
    pilot_technical = technical_tables(
        pilot_observation_sets, pilot_title_sets, list(pilot_jobs.items())
    )
    # Counts/coverage remain separate from the two end-to-end accuracy figures.
    pilot_counts = pilot_technical["accuracy_and_coverage_percent"].drop(
        columns=[
            "end_to_end_strict_accuracy",
            "prediction_only_strict_accuracy",
            "verifier_accepted_strict_accuracy",
        ]
    )
    pilot_export(pilot_counts, "counts_coverage_rejections")
    for name in ["verifier", "error_attempts", "task_time_minutes"]:
        pilot_export(pilot_technical[name], name)

    prompt_hit_status = "disabled"
    if os.getenv("PROMPT_HIT_DIAGNOSTICS", "1") != "0":
        prompt_hit_status = "report_only; no exclusions or rescoring"
        prompts = pilot_observations.drop_duplicates(
            ["style", "condition", "dataset_id", "item_key", "prompt_seed"]
        ).copy()
        prompts["title_in_prompt"] = pd.array(
            [
                None if pd.isna(text) else title_occurs_in_text(title, text)
                for title, text in zip(
                    prompts.expected_title, prompts.prompt_text, strict=True
                )
            ],
            dtype="boolean",
        )
        prompt_counts = (
            pilot_scopes(prompts)
            .groupby(["style", "model_pair", "domain"], sort=False)
            .agg(
                planned_prompts=("prompt_seed", "size"),
                available_prompts=("title_in_prompt", "count"),
                prompt_hits=("title_in_prompt", "sum"),
            )
            .reset_index()
        )
        prompt_counts["missing_prompts"] = (
            prompt_counts.planned_prompts - prompt_counts.available_prompts
        )
        pilot_export(prompt_counts, "prompt_hit_counts")
        pilot_export(
            prompts[
                [
                    "style",
                    "condition",
                    "dataset_id",
                    "item_key",
                    "domain",
                    "expected_title",
                    "prompt_seed",
                    "prompt_text",
                    "title_in_prompt",
                ]
            ],
            "prompt_hits",
        )
    print("Prompt-hit diagnostic:", prompt_hit_status)

### Style pilot — full candidate-pool illustratability

The pool is described separately; the analysis does not select titles or alter the pilot.


In [ ]:
if ANALYSIS_MODE == "style_pilot":
    candidate_job, candidate_runs = pilot_load_job(
        RATING_JOB, "Candidate pool", [f"rating_{model}" for model in QG]
    )
    candidate_status = {
        "status": f"unavailable: {pilot_availability[-1]['status']}",
        "expected_candidates": 900,
    }
    if candidate_job is None:
        print(
            "No candidate-pool histogram: no completed RATING_JOB. No 900-title distribution is inferred from selected titles or reconstruction jobs."
        )
    else:
        candidate_keys = ["dataset_id", "domain", "item_key"]
        # The persisted full-pool roster includes candidates with no rating rows.
        config = load_effective_config(
            next(iter(candidate_runs.values())) / EFFECTIVE_CONFIG_FILENAME
        )
        candidate_roster = pd.DataFrame(
            [
                {
                    "dataset_id": config.dataset.dataset_id,
                    "domain": item.domain,
                    "item_key": item.id,
                    "title": item.title,
                }
                for item in config.dataset.items
            ]
        )
        if len(candidate_roster) != 900 or candidate_roster.groupby(
            "domain"
        ).size().to_dict() != dict.fromkeys(DOMAINS, 300):
            raise ValueError(
                "RATING_JOB must cover all 900 candidates, 300 per domain, not a selected 90-title set."
            )
        ratings = candidate_job.ratings.reindex(
            columns=["entry_name", *candidate_keys, "title", "score"]
        ).copy()
        ratings["model"] = ratings.entry_name.astype("string").str.removeprefix(
            "rating_"
        )
        numeric = pd.to_numeric(ratings.score, errors="coerce")
        ratings["valid"] = numeric.between(0, 100) & numeric.mod(1).eq(0)
        ratings["score"] = numeric.where(ratings.valid)
        candidate_scores = ratings.pivot(
            index=candidate_keys, columns="model", values="score"
        ).reindex(
            index=pd.MultiIndex.from_frame(candidate_roster[candidate_keys]), columns=QG
        )
        candidate_scores["valid_model_ratings"] = candidate_scores.notna().sum(axis=1)
        candidate_scores["mean_score"] = candidate_scores[QG].mean(axis=1, skipna=False)
        candidate_scores = candidate_roster.merge(
            candidate_scores, on=candidate_keys, validate="one_to_one"
        )
        candidate_counts = (
            candidate_scores.groupby("domain")
            .mean_score.agg(candidates="size", complete_four_model_means="count")
            .reindex(DOMAINS)
        )
        candidate_counts["missing_means"] = (
            candidate_counts.candidates - candidate_counts.complete_four_model_means
        )
        rating_counts = (
            ratings.groupby(["domain", "model"])
            .valid.agg(recorded_ratings="size", valid_ratings="sum")
            .reindex(
                pd.MultiIndex.from_product([DOMAINS, QG], names=["domain", "model"]),
                fill_value=0,
            )
        )
        rating_counts["planned_ratings"] = 300
        rating_counts["missing_ratings"] = (
            rating_counts.planned_ratings - rating_counts.recorded_ratings
        )
        rating_counts["invalid_ratings"] = (
            rating_counts.recorded_ratings - rating_counts.valid_ratings
        )
        pilot_export(candidate_scores, "candidate_pool_scores")
        pilot_export(candidate_counts.reset_index(), "candidate_pool_counts")
        pilot_export(rating_counts.reset_index(), "candidate_pool_model_counts")
        bins = np.arange(0, 101, 10)
        histogram_rows = []
        fig, axes = plt.subplots(
            1, 3, figsize=(12, 4), sharex=True, sharey=True, layout="constrained"
        )
        for domain, ax in zip(DOMAINS, axes):
            scores = candidate_scores.loc[
                candidate_scores.domain.eq(domain), "mean_score"
            ].dropna()
            counts, _ = np.histogram(scores, bins=bins)
            histogram_rows.extend(
                {
                    "domain": domain,
                    "bin_left": int(left),
                    "bin_right": int(right),
                    "right_inclusive": bool(right == 100),
                    "titles": int(count),
                }
                for left, right, count in zip(bins[:-1], bins[1:], counts, strict=True)
            )
            ax.hist(scores, bins=bins, color=DOMAINS[domain][0], edgecolor="white")
            n = candidate_counts.loc[domain]
            ax.set(
                title=f"{domain.title()}\nN={int(n.candidates)}; complete={int(n.complete_four_model_means)}; missing={int(n.missing_means)}",
                xlabel="Equal four-model mean (0–100)",
                ylabel="Candidates",
                xlim=(0, 100),
            )
            ax.yaxis.set_major_locator(MaxNLocator(integer=True))
        pilot_save(
            fig,
            "candidate_pool_distribution",
            "Candidate-pool illustratability by domain",
        )
        pilot_export(pd.DataFrame(histogram_rows), "candidate_pool_histogram")
        candidate_status = {
            "status": "available",
            "population": "all 900 candidates; separate from the 90-title pilot and main samples",
            "candidates": len(candidate_scores),
            "complete_means": int(candidate_scores.mean_score.notna().sum()),
            "missing_means": int(candidate_scores.mean_score.isna().sum()),
            "method": "equal Q25/G3/Q38/G4 mean; four valid integer ratings (0–100) required; no imputation",
            "histogram_bins": bins.tolist(),
            "bin_convention": "[left, right), last bin includes 100",
        }

In [ ]:
if ANALYSIS_MODE == "style_pilot":
    pilot_export(pd.DataFrame(pilot_availability), "availability")
    notebook_path = next(
        (
            path
            for path in [Path("final_study.ipynb"), Path("notebooks/final_study.ipynb")]
            if path.is_file()
        ),
        None,
    )
    pilot_manifest = {
        "analysis_mode": ANALYSIS_MODE,
        "analysis_version": PILOT_VERSION,
        "review_only": True,
        "created_at_utc": datetime.now(UTC).isoformat(),
        "notebook": str(notebook_path.resolve()) if notebook_path else None,
        "notebook_sha256": pilot_hash(notebook_path) if notebook_path else None,
        "python": platform.python_version(),
        "packages": {
            name: version(name)
            for name in [
                "semantic-roundtrip",
                "pandas",
                "numpy",
                "matplotlib",
                "seaborn",
            ]
        },
        "scoring_source_sha256": pilot_hash(evaluation.__file__),
        "methods": {
            "strict": EXACT_MATCH_METHOD,
            "normalized": NORMALIZED_EXACT_METHOD,
        },
        "denominator": "all planned direct observations for BOTH metrics; rejected/missing/failed score zero",
        "aggregation": "one prompt seed × one image seed per title; equal title and diagonal-model-pair weights",
        "scope": pilot_scope,
        "availability": pilot_availability,
        "intervals": {
            "method": "paired whole-title percentile bootstrap, domain-stratified",
            "repetitions": 10000,
            "seed": 20260829,
            "coverage": 0.95,
            "pointwise": True,
        },
        "prompt_hits": {
            "status": prompt_hit_status,
            "function": "title_occurs_in_text",
            "method": prompt_hit_method,
            "exclusions": False,
        },
        "candidate_pool": candidate_status,
        "sources": pilot_sources,
        "exports": pilot_exports,
    }
    (OUTPUT_DIR / "style_pilot_manifest.json").write_text(
        json.dumps(pilot_manifest, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )
    display(JSON(pilot_manifest))
    print(
        "Style-pilot review complete. Full-study sections below are inactive in this mode."
    )

## Full-study analysis — sections A–E

The following code runs only in `ANALYSIS_MODE=full_study` (the default). In `style_pilot` mode,
these cells are deliberately inactive and produce no full-study results.


## A — Direct job

### Load once

Run this cell for the 4×4 analysis and the paired-route comparison. No indirect jobs are needed.


In [ ]:
if ANALYSIS_MODE == "full_study":
    direct = load_job(DIRECT_JOB)
    direct_obs = annotate(direct.observations)
    direct_titles = aggregate_titles(direct_obs, condition_columns=["pg", "bb", "bi"])
    direct_only = direct_titles[direct_titles.route == "direct"]
    print(
        direct_obs.dataset_id.iloc[0],
        "—",
        direct_only.condition.nunique(),
        "direct conditions",
    )

### RQ1 — Family, generation and PG/BI combination

**Question:** How do model family, model generation, and the combination of prompt-generation
and image-interpretation models relate to direct end-to-end title-reconstruction accuracy
for the selected Qwen and Gemma models?

Start with the complete 4×4 heatmap, then examine the four planned 2×2 comparisons.
For cells A=XX, B=XY, C=YX, D=YY:
PG = (C+D−A−B)/2; BI = (B+D−A−C)/2; same−mixed = (A+D−B−C)/2.
The last is a PG×BI interaction contrast, not an independent additional mechanism.
BI comparisons reuse images; PG comparisons legitimately use different generated images.


In [ ]:
if ANALYSIS_MODE == "full_study":
    direct_cells = 100 * direct_only.groupby(["pg", "bi"])[
        METRIC
    ].mean().unstack().reindex(index=QG, columns=QG)
    display(direct_cells)
    fig, ax = plt.subplots(figsize=(6, 5), layout="constrained")
    fig.colorbar(
        heatmap(ax, direct_only, QG, ""),
        ax=ax,
        label="End-to-end Strict Exact Match (%)",
    )
    save(fig, "direct_4x4", "RQ1: Direct reconstruction across PG and BI models")

In [ ]:
if ANALYSIS_MODE == "full_study":
    direct_contrasts = {}
    for label, (x, y) in PAIRS.items():
        xx, xy, yx, yy = [
            f"direct_pg_{pg}_bi_{bi}" for pg, bi in [(x, x), (x, y), (y, x), (y, y)]
        ]
        pair_label = f"{label} ({y.upper()} − {x.upper()})"
        direct_contrasts[f"PG: {pair_label}"] = difference([yx, yy], [xx, xy])
        direct_contrasts[f"BI: {pair_label}"] = difference([xy, yy], [xx, yx])
        direct_contrasts[f"Same − mixed: {label}"] = difference([xx, yy], [xy, yx])
    direct_effects = effects(direct_only, direct_contrasts)
    display(direct_effects[direct_effects.domain == "all"])
    roles = ["PG", "BI", "Same − mixed"]
    panel_titles = [
        "Prompt generation (PG)",
        "Image interpretation (BI)",
        "Model pairing (same - mixed)",
    ]
    comparison_colors = {
        "Qwen": "#0072B2",
        "Gemma": "#D55E00",
        "Family 2025": "#009E73",
        "Family 2026": "#CC79A7",
    }
    overall = direct_effects[direct_effects.domain == "all"]
    x_min = 5 * np.floor((overall.ci95_low.min() - 3) / 5)
    x_max = 5 * np.ceil((overall.ci95_high.max() + 3) / 5)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.8), sharex=True, layout="constrained")
    for role, title, ax in zip(roles, panel_titles, axes):
        table = direct_effects[
            (direct_effects.domain == "all")
            & direct_effects.comparison.str.startswith(role + ":")
        ].copy()
        table["comparison"] = table.comparison.str.removeprefix(
            role + ": "
        ).str.replace("−", "-", regex=False)
        interval_plot(ax, table, colors=list(comparison_colors.values()))
        ax.set(title=title, xlim=(x_min, x_max))
    fig.supxlabel(
        "Positive PG/BI values favour the first model in each subtraction; "
        "positive pairing values favour same-model pairs.",
        fontsize=9,
        color="#444444",
    )
    save(
        fig,
        "direct_primary_effects",
        "RQ1: Direct reconstruction - planned model contrasts",
    )

The four relation groups below each contain four cells. This secondary overview includes
combinations changing both family and generation; it is not an isolated family/age effect.


In [ ]:
if ANALYSIS_MODE == "full_study":
    family = {"q25": "qwen", "q38": "qwen", "g3": "gemma", "g4": "gemma"}
    cohort = {"q25": 2025, "g3": 2025, "q38": 2026, "g4": 2026}
    relation = np.select(
        [
            direct_only.pg.eq(direct_only.bi),
            direct_only.pg.map(family).eq(direct_only.bi.map(family)),
            direct_only.pg.map(cohort).eq(direct_only.bi.map(cohort)),
        ],
        [
            "Same model",
            "Same family, other generation",
            "Same generation, other family",
        ],
        default="Different family and generation",
    )
    direct_relations = (
        100 * direct_only.assign(relation=relation).groupby("relation")[METRIC].mean()
    ).rename("accuracy_percent")
    display(direct_relations.to_frame())

### RQ3 — Paired reconstruction routes

**Question:** For the same generated images, how does direct image-to-title reconstruction
differ from reconstruction through an explicit image description in the selected multimodal
baseline conditions?

Only the four diagonal conditions provide this comparison (BB=BI). Show absolute accuracies
next to description−direct differences: four baselines and their equal-weight mean.
A route difference does not by itself prove information loss. Transition counts below
describe images; the confidence intervals still use titles as the statistical unit.


In [ ]:
if ANALYSIS_MODE == "full_study":
    diagonal = direct_titles[direct_titles.pg == direct_titles.bi]
    route_means = (
        100 * diagonal.groupby(["pg", "route"])[METRIC].mean().unstack()
    ).reindex(QG)
    route_scores = diagonal.assign(
        condition_route=diagonal.condition + "__" + diagonal.route
    )
    route_contrasts = {
        m.upper(): difference(
            [f"direct_pg_{m}_bi_{m}__description"], [f"direct_pg_{m}_bi_{m}__direct"]
        )
        for m in QG
    }
    route_contrasts["Mean of four baselines"] = difference(
        [f"direct_pg_{m}_bi_{m}__description" for m in QG],
        [f"direct_pg_{m}_bi_{m}__direct" for m in QG],
    )
    route_effects = effects(
        route_scores, route_contrasts, condition_column="condition_route"
    )
    display(route_means, route_effects[route_effects.domain == "all"])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
    for route, marker, offset in [("direct", "o", -0.08), ("description", "s", 0.08)]:
        axes[0].plot(
            np.arange(4) + offset,
            route_means[route],
            marker=marker,
            linestyle="none",
            label=route,
        )
    axes[0].set(
        xticks=range(4),
        xticklabels=[m.upper() for m in QG],
        xlabel="Baseline (PG=BB=BI)",
        ylabel="End-to-end Strict Exact Match (%)",
        ylim=(-2, 102),
    )
    axes[0].legend()
    interval_plot(axes[1], route_effects[route_effects.domain == "all"])
    axes[0].set_title("Absolute accuracy")
    axes[1].set_title("Description - direct")
    save(fig, "paired_routes", "RQ3: Direct and description-mediated reconstruction")

    transitions = (
        direct_obs[direct_obs.pg == direct_obs.bi]
        .pivot(
            index=["condition", "domain", "item_key", "prompt_seed", "image_seed"],
            columns="route",
            values="end_to_end_strict_score",
        )
        .reset_index()
    )
    transition_order = [
        "Both correct",
        "Direct only",
        "Description only",
        "Neither correct",
    ]
    transitions["outcome"] = np.select(
        [
            transitions.direct.eq(1) & transitions.description.eq(1),
            transitions.direct.eq(1),
            transitions.description.eq(1),
        ],
        transition_order[:3],
        default=transition_order[3],
    )
    route_transitions = (
        transitions.groupby(["condition", "domain", "outcome"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=transition_order, fill_value=0)
    )
    display(route_transitions.groupby("condition").sum())

### Prepare domain-specific direct results

**Question:** How do reconstruction results differ between songs, movie titles, and band names?

RQ4 is presented once, across both designs, in section C. This cell only prepares the
domain-specific planned contrasts used there. Different domains contain different titles
and are not paired with each other.


In [ ]:
if ANALYSIS_MODE == "full_study":
    direct_domain_effects = direct_effects[direct_effects.domain != "all"].copy()
    route_domain_effects = route_effects[route_effects.domain != "all"].copy()

### Exploratory direct illustratability

Correlations use each PG model's rating and accuracy averaged across all four BI models, direct route only.
The three domain histograms give each title one value: the equal mean of its Q25, G3, Q38 and G4 ratings
from the Direct job. All four ratings are required; missing means are reported separately.
Technical validity is consolidated in section C.


In [ ]:
if ANALYSIS_MODE == "full_study":
    title_characteristics = direct_obs[
        ["dataset_id", "item_key", "domain", "expected_title", "title_length_group"]
    ].drop_duplicates()
    title_length_counts = (
        title_characteristics.groupby(["domain", "title_length_group"])
        .size()
        .unstack(fill_value=0)
        .reindex(index=DOMAINS, columns=["short", "medium", "long"], fill_value=0)
    )
    display(title_length_counts)
    fig, ax = plt.subplots(figsize=(7, 3.5), layout="constrained")
    sns.countplot(
        data=title_characteristics,
        x="domain",
        hue="title_length_group",
        hue_order=["short", "medium", "long"],
        ax=ax,
    )
    ax.set(xlabel="Domain", ylabel="Titles")
    save(fig, "title_length_distribution", "Title-length distribution by domain")

    direct_rating_summary, direct_rating_pairs = rating_analysis(
        direct.ratings,
        direct_only,
        "direct_illustratability",
        "SQ3: Illustratability and direct reconstruction",
    )
    display(
        direct_rating_summary.style.format(
            {column: "{:.2f}" for column in ["spearman_rho", "ci95_low", "ci95_high"]},
            na_rep="n/a",
        )
    )

    rating_distribution = direct.ratings.reindex(
        columns=["entry_name", "dataset_id", "domain", "item_key", "score"]
    ).copy()
    rating_distribution["pg"] = rating_distribution.entry_name.str.extract(
        r"_pg_([^_]+)_", expand=False
    )
    rating_index = ["dataset_id", "domain", "item_key"]
    rating_means = (
        rating_distribution.drop_duplicates([*rating_index, "pg"])
        .pivot(index=rating_index, columns="pg", values="score")
        .reindex(columns=QG)
        .mean(axis=1, skipna=False)
        .rename("mean_score")
    )
    rating_means = title_characteristics[rating_index].merge(
        rating_means, on=rating_index, how="left"
    )
    rating_counts = (
        rating_means.groupby("domain")
        .mean_score.agg(titles="size", complete_ratings="count")
        .reindex(DOMAINS)
    )
    rating_counts["missing_ratings"] = (
        rating_counts.titles - rating_counts.complete_ratings
    )
    display(rating_counts)
    fig, axes = plt.subplots(
        1, 3, figsize=(12, 3.8), sharex=True, sharey=True, layout="constrained"
    )
    for domain, ax in zip(DOMAINS, axes):
        sns.histplot(
            data=rating_means[rating_means.domain == domain],
            x="mean_score",
            color=DOMAINS[domain][0],
            bins=np.arange(0, 101, 10),
            ax=ax,
        )
        ax.set(
            title=f"{domain.title()} (n={rating_counts.loc[domain, 'complete_ratings']})",
            xlabel="Mean illustratability (0–100)",
            ylabel="Titles",
            xlim=(0, 100),
        )
        ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    save(
        fig,
        "illustratability_distribution",
        "SQ3: Model-averaged illustratability by domain",
    )

## B — Complete indirect 3×4×3 matrix

### Load once

Load the completed local 16-condition job and its complete 20-condition Aqueduct extension.
Together they form the 36-cell matrix. The hosted extension must inherit from this local job.


In [ ]:
if ANALYSIS_MODE == "full_study":
    local = load_job(INDIRECT_JOB)
    aqueduct = load_job(AQUEDUCT_JOB)
    local_obs = annotate(local.observations)
    aqueduct_obs = annotate(aqueduct.observations)
    local_titles = aggregate_titles(local_obs, condition_columns=["pg", "bb", "bi"])
    aqueduct_titles = aggregate_titles(
        aqueduct_obs, condition_columns=["pg", "bb", "bi"]
    )
    full_obs = pd.concat([local_obs, aqueduct_obs], ignore_index=True)
    full_titles = pd.concat([local_titles, aqueduct_titles], ignore_index=True)
    full_ratings = pd.concat([local.ratings, aqueduct.ratings], ignore_index=True)
    print(
        full_obs.dataset_id.iloc[0],
        "—",
        local_titles.condition.nunique(),
        "local +",
        aqueduct_titles.condition.nunique(),
        "hosted =",
        full_titles.condition.nunique(),
        "indirect conditions",
    )

### RQ2 — PG/BB/BI combinations

**Question:** How do the selection and combination of the prompt-generation, image-description,
and description-interpretation models relate to title-reconstruction accuracy on the
description-mediated route?

Four BB panels show the complete 3×4×3 matrix. The six local effects retain their
predeclared D32/O120 scope. Two separately labelled hosted effects compare V4 with the
mean of D32 and O120 over the full matrix. Matching is supporting, not the definition of RQ2.
These are deployed-configuration comparisons, not isolated architecture or age effects.


In [ ]:
if ANALYSIS_MODE == "full_study":
    bb_heatmaps(
        full_titles,
        TEXT,
        "indirect_3x4x3_V4_hosted",
        "RQ2: Description-mediated reconstruction (V4 externally hosted)",
    )
    local_effects = effects(local_titles, indirect_contrasts(local_titles)).assign(
        scope="D32/O120 submatrix"
    )
    hosted_contrasts = {
        name: weights
        for name, weights in indirect_contrasts(full_titles).items()
        if "V4" in name
    }
    hosted_effects = effects(full_titles, hosted_contrasts).assign(
        scope="Full 3×4×3 matrix"
    )
    indirect_effects = pd.concat([local_effects, hosted_effects], ignore_index=True)
    display(indirect_effects[indirect_effects.domain == "all"])
    plot_effects = indirect_effects[indirect_effects.domain == "all"].copy()
    plot_effects["comparison"] = plot_effects.scope + ": " + plot_effects.comparison
    fig, ax = plt.subplots(figsize=(9, 5), layout="constrained")
    interval_plot(ax, plot_effects)
    save(
        fig,
        "indirect_primary_effects",
        "RQ2: Description-mediated reconstruction - model-role contrasts",
    )
    local_matching = effects(local_titles, matching_contrasts(local_titles)).assign(
        scope="D32/O120 submatrix"
    )
    full_matching = effects(full_titles, matching_contrasts(full_titles)).assign(
        scope="Full 3×4×3 matrix"
    )
    indirect_matching = pd.concat([local_matching, full_matching], ignore_index=True)
    display(indirect_matching[indirect_matching.domain == "all"])

### Exploratory indirect illustratability

Every PG rating is related to accuracy over the same four BB × three BI downstream set.
Domain results and technical validity are presented once in section C.


In [ ]:
if ANALYSIS_MODE == "full_study":
    indirect_domain_effects = indirect_effects[indirect_effects.domain != "all"].copy()
    indirect_domain_matching = indirect_matching[
        indirect_matching.domain != "all"
    ].copy()
    full_rating_summary, full_rating_pairs = rating_analysis(
        full_ratings,
        full_titles,
        "indirect_illustratability",
        "SQ3: Illustratability and description-mediated reconstruction",
    )
    display(
        full_rating_summary.style.format(
            {column: "{:.2f}" for column in ["spearman_rho", "ci95_low", "ci95_high"]},
            na_rep="n/a",
        )
    )

## C — RQ4: Domain comparison

Run sections A and B first. This section combines no accuracy across designs; it presents
domain means side by side. Section E consolidates diagnostics after loading supplements.


In [ ]:
if ANALYSIS_MODE == "full_study":

    def overall_domain_means(frame, design):
        conditions = frame.condition.unique()
        table = effects(
            frame, {"Mean accuracy": dict.fromkeys(conditions, 1 / len(conditions))}
        )
        return table[table.domain != "all"].assign(design=design)

    direct_domains = overall_domain_means(direct_only, "Direct 4×4")
    indirect_domains = overall_domain_means(full_titles, "Indirect 3×4×3")
    domain_results = pd.concat([direct_domains, indirect_domains], ignore_index=True)

### RQ4 — Songs, movies and bands

The figure averages conditions equally within each design and shows the two designs separately.
Different domains contain different titles, so the intervals are domain-specific title-sampling
uncertainty, not paired domain contrasts. Domain-specific planned effects remain available in
`direct_domain_effects`, `route_domain_effects`, and `indirect_domain_effects`.


In [ ]:
if ANALYSIS_MODE == "full_study":
    display(domain_results)
    fig, ax = plt.subplots(figsize=(7, 3.5), layout="constrained")
    offsets = {"Direct 4×4": -0.10, "Indirect 3×4×3": 0.10}
    markers = {"Direct 4×4": "o", "Indirect 3×4×3": "s"}
    for design, table in domain_results.groupby("design", sort=False):
        table = table.set_index("domain").reindex(DOMAINS)
        y = np.arange(len(DOMAINS)) + offsets[design]
        errors = np.vstack(
            [table.estimate - table.ci95_low, table.ci95_high - table.estimate]
        )
        ax.errorbar(
            table.estimate, y, xerr=errors, fmt=markers[design], capsize=2, label=design
        )
    ax.set(
        yticks=range(len(DOMAINS)),
        yticklabels=list(DOMAINS),
        xlim=(0, 100),
        xlabel="End-to-end Strict Exact Match (%), pointwise 95% CI",
    )
    ax.legend()
    save(fig, "domain_overview", "RQ4: Reconstruction accuracy by domain")

## D — Direct supplementary analyses

These analyses do not change the primary random, unrestricted, thinking-off study. Sketch and
comic alter only the PG style instruction. The thinking supplement changes native thinking for
Q38/G4 wherever either model occupies PG or BI; the verifier remains thinking-off. The
high-illustratability supplement changes the selected titles and is therefore descriptive rather
than a paired population comparison.

In [ ]:
if ANALYSIS_MODE == "full_study":

    def load_direct_supplement(path, label):
        if path is None:
            print(f"{label} path is not set; analysis skipped.")
            return None, None, None
        job = load_job(path)
        observations = annotate(job.observations)
        titles = aggregate_titles(observations, condition_columns=["pg", "bb", "bi"])
        direct_scores = titles[titles.route == "direct"]
        print(
            observations.dataset_id.iloc[0],
            "—",
            direct_scores.condition.nunique(),
            f"{label} direct conditions",
        )
        return job, observations, direct_scores

    sketch, sketch_obs, sketch_direct = load_direct_supplement(SKETCH_JOB, "sketch")
    comic, comic_obs, comic_direct = load_direct_supplement(COMIC_JOB, "comic")

### SQ1 — Paired sketch and comic comparisons

Each comparison shows two accuracy matrices and their difference in percentage points:
Sketch - unrestricted, Comic - unrestricted, and Sketch - Comic. Positive differences favour
the first named condition. Paired effect plots summarise all 16 cells and, separately, each
PG or BI model averaged over its four counterpart models. All three comparisons use the same
titles, seeds, models and image settings. Their 95% intervals are pointwise, not adjusted
across comparisons. These are robustness analyses, not a replication of the human sketch study.

In [ ]:
if ANALYSIS_MODE == "full_study":

    def paired_style_analysis(positive, negative, label, reference_label, slug):
        if positive is None or negative is None:
            return None
        comparison = f"{label} - {reference_label}"
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), layout="constrained")
        image = heatmap(axes[0], negative, QG, reference_label)
        heatmap(axes[1], positive, QG, label)
        delta = heatmap(axes[2], positive, QG, comparison, baseline=negative)
        fig.colorbar(
            image, ax=list(axes[:2]), label="End-to-end Strict Exact Match (%)"
        )
        fig.colorbar(delta, ax=axes[2], label="Difference (pp)")
        save(fig, f"direct_{slug}_matrices", f"SQ1: {comparison}")

        reference_scores = negative.assign(
            condition_style=lambda f: "reference__" + f.condition.astype(str)
        )
        comparison_scores = positive.assign(
            condition_style=lambda f: "comparison__" + f.condition.astype(str)
        )
        combined = pd.concat([reference_scores, comparison_scores], ignore_index=True)
        contrasts = {
            f"Overall: {comparison}": difference(
                comparison_scores.condition_style.unique(),
                reference_scores.condition_style.unique(),
            )
        }
        for role in ["pg", "bi"]:
            for model in QG:
                contrasts[f"{role.upper()}={model.upper()}: {comparison}"] = difference(
                    comparison_scores.loc[
                        comparison_scores[role] == model, "condition_style"
                    ].unique(),
                    reference_scores.loc[
                        reference_scores[role] == model, "condition_style"
                    ].unique(),
                )
        table = effects(combined, contrasts, condition_column="condition_style")
        display(table[table.domain == "all"])
        plotted = table[table.domain == "all"].copy()
        plotted["comparison"] = plotted.comparison.str.split(":").str[0]
        fig, ax = plt.subplots(figsize=(9, 6), layout="constrained")
        interval_plot(ax, plotted)
        save(fig, f"direct_{slug}_effects", f"SQ1: {comparison}")
        return table

    sketch_effects = paired_style_analysis(
        sketch_direct, direct_only, "Outline sketch", "Unrestricted", "sketch"
    )
    comic_effects = paired_style_analysis(
        comic_direct, direct_only, "Comic/cartoon", "Unrestricted", "comic"
    )
    sketch_comic_effects = paired_style_analysis(
        sketch_direct,
        comic_direct,
        "Outline sketch",
        "Comic/cartoon",
        "sketch_vs_comic",
    )

### SQ2 — Native-thinking deployment comparison

The native-thinking matrix uses the twelve newly calculated cells plus the four unchanged
Q25/G3 cells from the unrestricted Direct job. Absolute matrices and a native-minus-off
difference heatmap are accompanied by paired effects for all 16 cells, the 12 changed cells,
and three disjoint four-cell groups: native thinking in PG only, BI only, or both roles.
The groups contain different PG/BI pairs; their differences do not isolate a role interaction.
This is a deployment-policy comparison, including changed output ceilings and timeouts,
not an isolated causal effect of thinking or equal test-time compute.

In [ ]:
if ANALYSIS_MODE == "full_study":
    if THINKING_JOB is None:
        thinking = thinking_obs = thinking_direct = thinking_effects = None
        print("THINKING_JOB is not set; native-thinking analysis skipped.")
    else:
        thinking, thinking_obs, thinking_changed = load_direct_supplement(
            THINKING_JOB, "native-thinking changed-cell"
        )
        unchanged = direct_only[
            direct_only.pg.isin(["q25", "g3"]) & direct_only.bi.isin(["q25", "g3"])
        ]
        thinking_direct = pd.concat([unchanged, thinking_changed], ignore_index=True)
        if thinking_direct.condition.nunique() != 16:
            raise ValueError(
                "Thinking matrix does not contain 12 changed and 4 baseline cells."
            )

        fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), layout="constrained")
        image = heatmap(axes[0], direct_only, QG, "Thinking off")
        heatmap(axes[1], thinking_direct, QG, "Native thinking for Q38/G4")
        delta = heatmap(
            axes[2], thinking_direct, QG, "Native - off", baseline=direct_only
        )
        fig.colorbar(
            image, ax=list(axes[:2]), label="End-to-end Strict Exact Match (%)"
        )
        fig.colorbar(delta, ax=axes[2], label="Difference (pp)")
        save(fig, "direct_thinking_matrices", "SQ2: Native thinking")

        off_scores = direct_only.assign(
            condition_mode=lambda f: "off__" + f.condition.astype(str)
        )
        native_scores = thinking_direct.assign(
            condition_mode=lambda f: "native__" + f.condition.astype(str)
        )
        combined = pd.concat([off_scores, native_scores], ignore_index=True)
        cells = thinking_direct[["condition", "pg", "bi"]].drop_duplicates()
        native_pg = cells.pg.isin(["q38", "g4"])
        native_bi = cells.bi.isin(["q38", "g4"])
        thinking_groups = {
            "All 16 cells": cells.condition,
            "12 changed cells": cells.loc[native_pg | native_bi, "condition"],
            "PG only (4 cells)": cells.loc[native_pg & ~native_bi, "condition"],
            "BI only (4 cells)": cells.loc[~native_pg & native_bi, "condition"],
            "PG and BI (4 cells)": cells.loc[native_pg & native_bi, "condition"],
        }
        thinking_contrasts = {
            f"{label}: native - off": difference(
                native_scores.loc[
                    native_scores.condition.isin(conditions), "condition_mode"
                ].unique(),
                off_scores.loc[
                    off_scores.condition.isin(conditions), "condition_mode"
                ].unique(),
            )
            for label, conditions in thinking_groups.items()
        }
        thinking_effects = effects(
            combined, thinking_contrasts, condition_column="condition_mode"
        )
        display(thinking_effects[thinking_effects.domain == "all"])
        plotted = thinking_effects[thinking_effects.domain == "all"].copy()
        plotted["comparison"] = plotted.comparison.str.split(":").str[0]
        fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
        interval_plot(ax, plotted)
        save(
            fig,
            "direct_thinking_effects",
            "SQ2: Native thinking - thinking off",
        )

### SQ3 — Random versus high-illustratability titles

The two datasets contain different titles. The matrices and differences below are descriptive
and do not estimate a paired, causal or population-wide effect of filtering. In particular,
development results contain only six titles per dataset and demonstrate the analysis path.

In [ ]:
if ANALYSIS_MODE == "full_study":
    if ILLUSTRATABLE_JOB is None:
        illustratable = illustratable_obs = illustratable_direct = None
        print("ILLUSTRATABLE_JOB is not set; high-illustratability analysis skipped.")
    else:
        illustratable, illustratable_obs, illustratable_direct = load_direct_supplement(
            ILLUSTRATABLE_JOB, "high-illustratability"
        )
        fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), layout="constrained")
        image = heatmap(axes[0], direct_only, QG, "Random titles")
        heatmap(axes[1], illustratable_direct, QG, "High illustratability")
        delta = heatmap(
            axes[2], illustratable_direct, QG, "Selected - random", baseline=direct_only
        )
        fig.colorbar(
            image, ax=list(axes[:2]), label="End-to-end Strict Exact Match (%)"
        )
        fig.colorbar(delta, ax=axes[2], label="Difference (pp)")
        save(
            fig,
            "direct_illustratability_matrices",
            "SQ3: High-illustratability vs random titles (descriptive)",
        )
        descriptive_dataset_comparison = pd.DataFrame(
            {
                "dataset": ["Random", "High illustratability"],
                "titles": [
                    direct_only.item_key.nunique(),
                    illustratable_direct.item_key.nunique(),
                ],
                "mean_accuracy_percent": [
                    100 * direct_only[METRIC].mean(),
                    100 * illustratable_direct[METRIC].mean(),
                ],
            }
        )
        display(descriptive_dataset_comparison)

## E — Technical validity and supporting accuracy

Run the preceding sections first; omitted supplements are skipped. Thinking includes the
four unchanged baseline cells in its accuracy, but does not count them as new computation.
Counts represent condition/route observations, not unique images or fresh model calls.
Rows with condition or domain `all` are pooled totals: do not add them to detail rows.
Every table is saved as CSV; undefined values are displayed as `n/a`.
Imported errors and timings are counted only once, at their original local source.


In [ ]:
if ANALYSIS_MODE == "full_study":
    observation_sets = {"Direct": direct_obs, "Indirect 3×4×3": full_obs}
    title_sets = {"Direct": direct_titles, "Indirect 3×4×3": full_titles}
    job_sets = [
        ("Direct", direct),
        ("Indirect 3×4×3", local),
        ("Indirect 3×4×3", aqueduct),
    ]
    for label, job, observations, scores in [
        ("Sketch", sketch, sketch_obs, sketch_direct),
        ("Comic", comic, comic_obs, comic_direct),
        ("Thinking", thinking, thinking_obs, thinking_direct),
        ("Illustratable", illustratable, illustratable_obs, illustratable_direct),
    ]:
        if job is None:
            continue
        observations = observations[observations.route.eq("direct")]
        if label == "Thinking":
            baseline = direct_obs[
                direct_obs.route.eq("direct")
                & direct_obs.pg.isin(["q25", "g3"])
                & direct_obs.bi.isin(["q25", "g3"])
            ]
            observations = pd.concat([observations, baseline], ignore_index=True)
        observation_sets[label] = observations
        title_sets[label] = scores
        job_sets.append((label, job))

    print("Strict Exact Match:", EXACT_MATCH_METHOD)
    print("Normalized Exact Match:", NORMALIZED_EXACT_METHOD)
    technical = technical_tables(observation_sets, title_sets, job_sets)
    for name, table in technical.items():
        print(name)
        table.to_csv(OUTPUT_DIR / f"{name}.csv", index=False, na_rep="n/a")
        display(table.style.format(precision=2, na_rep="n/a"))